# Segmentation model · DINOv3 ConvNeXt-Base

**Final architecture:** DINOv3 ConvNeXt-Base encoder, EdgeAttNet decoder, and two edge-guided attention modules. This notebook reproduces the selected **55-epoch EMA fine-tuning recipe**. It uses the fixed physical-image split and paired training targets bundled in this repository.

The default mode loads the selected inference weights. Fine-tuning is optional and starts from official pretrained DINOv3 encoder weights; it is not self-supervised pretraining from random initialization.

In [ ]:
# Locate the repository from its root or notebooks/ directory.
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p/'src'/'architecture.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the extracted repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)


## 1. Execution settings

Use the pretrained model for committee inspection. Set `TRAIN_FROM_SCRATCH = True` only to reproduce the complete segmentation fine-tuning run. A separate OOF process is required only when refitting the CatBoost filter.

In [ ]:
from pathlib import Path
import os, sys, subprocess

TRAIN_FROM_SCRATCH = False
EXPORT_INFERENCE_WEIGHTS = False
PRETRAINED_DINO_ENCODER = None  # Required only for fine-tuning; local trusted state_dict
MODEL_DIR = None  # Optional local final-artifact directory
OUTPUT_ROOT = Path('/kaggle/working/EdgeAttNet_DUAL_v1')

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / 'src' / 'architecture.py').is_file(), 'Open this notebook with the repository as working directory.'
sys.path.insert(0, str(REPO_ROOT))

## 2. Dataset integrity

The original **566 training / 141 validation** split is fixed. Semantic and spine training targets are restored from the bundled archive; no new split is sampled.

In [ ]:
from scripts.prepare_training_assets import prepare
from src.data import load_split

split_root = REPO_ROOT / 'data' / 'splits'
physical, annotation_sets = load_split(split_root)
print('Split metadata:', split_root)
print('Physical images:', len(physical), '| Annotation sets:', len(annotation_sets))
if TRAIN_FROM_SCRATCH:
    mask_root = prepare()
    print('Training targets:', mask_root)

## 3. Selected model and training

**Recipe:** 1024px training crops, balanced augmentation, AdamW with encoder-specific learning-rate scaling, mixed precision, 0.99 EMA, and fixed 55-epoch Dice-stage optimization. Checkpoint selection uses the original fixed decoder monitor on the tuning partition. The selected historical artifact was saved at epoch 50; a fresh run may select a different epoch.

No alternate encoders or model comparisons are executed here.

In [ ]:
if TRAIN_FROM_SCRATCH:
    if PRETRAINED_DINO_ENCODER is None:
        raise FileNotFoundError('Fine-tuning requires the official licensed DINOv3 ConvNeXt-Base encoder state_dict. Set PRETRAINED_DINO_ENCODER to its path.')
    encoder = Path(PRETRAINED_DINO_ENCODER)
    assert encoder.is_file(), f'Missing pretrained encoder: {encoder}'
    subprocess.run([sys.executable, str(REPO_ROOT/'scripts'/'retrain_base.py'), '--encoder', str(encoder), '--output-root', str(OUTPUT_ROOT)], check=True)
    selected = OUTPUT_ROOT/'dino_convnext_base'/'dice_best_ema.pth'
else:
    from src.artifacts import resolve_artifacts
    selected = resolve_artifacts(MODEL_DIR)['weights']
print('Selected model:', selected)

## 4. Export for inference

The committee uses a model-only state dictionary. Export is optional when the published model-only artifact already exists.

In [ ]:
if EXPORT_INFERENCE_WEIGHTS:
    target = OUTPUT_ROOT/'edgeattnet_dinov3_base_best.pth'
    subprocess.run([sys.executable, str(REPO_ROOT/'scripts'/'export_model.py'), '--source',str(selected),'--output',str(target)], check=True)
    print('Inference weights:', target)
else:
    print('Model is available. Run final_inference.ipynb for evaluation and CSV export.')